# Sport Wagering and Daily Fantasy Sports

"To be successful in sports betting, you need three things: brains, guts, and
money. If you are missing one of them, you're dead." -- Charles K. McNeil

## Python Modules

We will use the following Python modules in this chapter that we have used in
the past:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import isapy_data

We will also introduce the `PuLP` library for solving linear programming
(optimization) problems:

In [ ]:
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, value, PULP_CBC_CMD

In [ ]:
#| echo: false

sns.set_theme(style="whitegrid")

Note that in this chapter, rather than using a separate R package (`bettoR`)
for wagering calculations, we will write our own helper functions in Python.
This serves both as an educational exercise and as an illustration of how
straightforward the underlying math is.

## If you don't know anything about Sports Wagering

Sports wagering has likely existed since the beginning of athletic
competitions. Some of the earliest mentions of sports wagering refer to the
ancient Olympic Games in Greece. It is also widely suspected that some of those
earliest Olympians may have been paid to not perform their best, thus altering
the outcome of the events to the advantage of certain wagerers who were in the
know. It is also well known that the Romans wagered feverishly on the outcome
of chariot races and gladiator contests, and many other societies have since
adopted sports wagering.

Today, people can wager on just about any type of sport. Football is the most
wagered upon sport in the US, while in the rest of the world it is soccer and
tennis. But not only major sports see wagering. Cricket, darts, and lacrosse
are wagered upon. During the early days of the COVID-19 pandemic, table tennis
matches in Russia were close to the only athletic competitions being held
worldwide, and they were wagered upon aggressively in a world with very little
to do but sit at home in quarantine. The US was no exception to this, with a
2018 US Supreme Court decision paving the way for legal gambling to expand
across the US instead of simply Nevada. According to a Statista report at
[https://www.statista.com/statistics/1126480/sports-betting-revenue-us/](https://www.statista.com/statistics/1126480/sports-betting-revenue-us/),
the size of the legal sports betting market in the US went from \$430 million
in 2018 to a record \$16.96 billion in 2025, according to the American Gaming
Association[^wager-1], and continues to grow as sports betting becomes legal in
more jurisdictions. Moreover, the size of the underground/illegal betting
market in the USA is estimated to be 10 times the size of the legal market,
though of course no one knows for sure.

[^wager-1]: American Gaming Association, "Sports Betting Hits Record \$16.96
Billion Revenue in 2025,"
[https://www.espn.com/espn/betting/story/_/id/48045855/](https://www.espn.com/espn/betting/story/_/id/48045855/)

Sports wagering can be not only on individuals or teams, but often the outcomes
of animals racing, such as horses and dogs.

In this chapter, we will cover the various types of sports wagers that can be
made, probabilities vs. odds, and converting between the two. We will also
cover Daily Fantasy Sports and setting lineups in a separate subsection after
the wagering section.

## Sports Wagering and Morality

The authors of this text take no moral stance on sports wagering and attempt to
present it as such. We do not try to promote or dissuade it any more or less
than we might any other financial market. This market, in its simplest form,
represents the "buying" or "selling" of a team or player, rather than the
buying or selling of stocks or bonds in the traditional financial market. The
biggest difference between these markets and other financial markets is that
there is no "buy and hold" strategy, as the "accounts" are settled each day at
the conclusion of each sporting event. In that way it is more like a financial
options market where the markets settle each day. Like investing in any other
financial product, we recommend these methods be applied wisely and only where
legal.

## Sports Wagering and Legality

The legality of sports wagering varies around the world. In the UK for example,
it is a largely open market where many companies can each serve as market
makers, more commonly known as "bookies". In Finland, sports gambling is a
regulated market with one market maker (Veikkaus) which is owned by the
government. In other countries, wagering on sports is strictly illegal.
Countries with Islamic backgrounds, such as Qatar, Brunei, and the UAE are
perhaps the most salient examples of nations banning gambling of all forms, and
yet underground/illegal gambling exists even there despite the threat of strict
penalties if caught.

In the United States, each of the 50 states and the District of Columbia has
its own set of laws regarding sports betting. As of March 2026, 39 states and
Washington DC have some form of legalized sports betting. Among the most
populous states, California and Texas still do not have legalized sports
betting, so it remains unavailable legally to a significant share of the
population. Utah and Hawai'i currently have no legal gambling (lotteries,
casinos, horse racing, etc.) of any kind, and there is no reason to believe
that they will allow it anytime soon.

Even in places where sports wagering is licensed and regulated, there continue
to be illegal bookmakers. These bookmakers do virtually the same thing as the
licensed bookmakers, except they do not advertise and likely do not pay taxes
on their profits. Illegal bookmakers may also operate on credit, allowing
people to make wagers over the phone and then settling the bets perhaps each
week or each month or when one side has reached a limit of liability. These
services are sometimes operated by organized crime figures or their affiliates,
and have a reputation for nefarious ways of collecting on their debts.

## Basics of Sports Wagering

In its most basic form, individuals can wager on the outcome of a match. For
example, let's say that Roger Federer is playing a tennis match against Rafael
Nadal. While both are highly skilled tennis players, a sports wagering market
maker (often called a bookmaker, bookie, or sports book) calculates that
Federer is likely to win this match 75% of the time. The sports book offers to
accept bets on Federer as a 3 to 1 favorite (he will win three times for every
loss). This means that if a bettor wanted to place a wager that Federer would
win, they would have to risk \$3 in order to win \$1. This odds, or the market
price, can be expressed in a variety of ways which will be shown in detail
later.

If a bettor wanted to wager that Nadal would win the match, it would make
sense then that since he is not as likely to win as Federer that the return
should be greater. A completely fair market would then allow a bettor to win
\$3 for every \$1 risked on Nadal to win. However, it is extremely rare to find
a completely fair market, and it might be more likely that a bettor would win
\$2.25 for every dollar risked on Nadal to win.

## The role of the bookmaker

Why the discrepancy between the sides of the market as mentioned above? This
is essentially the fee that the bookmaker charges to provide the market
service. This differential is called juice, vig, or vigorish. The wider the
differential between the price on the two sides of the market, the more money
the bookmaker stands to make.

In [ ]:
#| fig-cap: "A typical scene in a sports book."
#| label: fig-book
#| echo: false

from IPython.display import Image, display
display(Image(filename="fig/board.jpeg", width=600))

Traditionally the role of the bookmaker is to set the price of the market such
that the outcome of the event does not matter and they will make the same
amount of money regardless of the outcome. To use the above example, the
bookmaker would like to attract \$123.08 of wagers on Nadal to win for every
\$300 that is attracted on Federer to win. If this happens, the bookmaker will
win \$23.08 no matter who wins.

In [ ]:
federer_exposure = 300
nadal_exposure = 123.08
bookmaker_profit_federer_wins = nadal_exposure - (federer_exposure / 3)
bookmaker_profit_nadal_wins = federer_exposure - (nadal_exposure * 2.25)
print(f"When Federer wins, the bookmaker profits ${bookmaker_profit_federer_wins:.2f}")
print(f"When Nadal wins, the bookmaker profits ${bookmaker_profit_nadal_wins:.2f}")

When a bookmaker starts to have more money attracted on one side of a market
(**e.g.**, Federer to win) than the odds might indicate, the bookmaker then has
a liability and as such a vested interest in the outcome of the match. In the
above example, let's say that the bookmaker takes \$300 in wagers for Federer
to win, but only \$50 on Nadal to win. If this happens, the bookmaker will keep
\$187.50 if Nadal wins, but will lose \$50 if Federer wins.

In [ ]:
# Nadal Wins
print(f"Nadal wins: ${300 - (50 * 2.25):.2f}")
# Federer Wins
print(f"Federer wins: ${50 - (300 / 3):.2f}")

The bookmaker of course cannot force people to bet on Nadal anymore than a
stock market maker can force people to buy a put option on Google stock instead
of a call option. So how does the bookmaker avoid having financial exposure to
the outcome, which is referred to as getting sided? Much like the stock market
maker, the bookmaker changes the prices to make bets on Federer seem less
attractive and bets on Nadal seem more attractive. The bookmaker might change
the price to require people to risk \$3.50 to win \$1 if the bettors prefer
Federer. At the same time the bookmaker may now offer a return of \$2.75 for
every \$1 risked on a wager for Nadal to win. This should cause the balance of
money wagered to flow away from Federer and towards Nadal.

It is important to note that sports betting is unlike parimutuel betting on
horses, where the odds for everyone change as the bets are placed up to the
event. In sports betting, the odds that are presented at the time of the event
are the odds for that person who made that bet. So in the above example, if a
bettor placed a bet on Federer at 3-to-1 odds, and Nadal twists his ankle in
warmups and the odds change to Federer being a 10-to-1 favorite just before
the match starts, that bettor would still receive payment at 3-to-1 odds.

In recent years, bookmakers have hired teams of analytics consultants and now
many are not just interested in balancing the market. They will have an opinion
on the outcome of a match, and may try to move the market prices in such a way
that the market appears tilted more towards one outcome than it might appear to
the general public. In essence, the bookmaker becomes like a bettor, gambling
in his own casino. We mention this not so much for teaching individuals how to
be a profitable bookmaker, but rather so the reader knows the contemporary
state of events.

More recently, exchanges which more resemble equity markets have begun to
appear in the sports betting world. In these exchanges, sometimes also called
peer-to-peer betting, the exchanges attempt to match buyers and sellers of a
particular market at the same price. To take the example above, an exchange
might offer Federer at 3 to 1 odds and Nadal at the exact opposite, 1 to 3
odds. The winner of the bet would then pay a small amount as a commission so
that the exchange operator can make a profit for operating the service. These
exchanges also allow bettors to offer a wide variety of bets and prices and see
if anyone is willing to take those bets.

## Popular Types of Wagers

### Moneyline {.unnumbered}

The simplest type of wager, which is just on one person or team to win an
event against another person or team, is referred to as moneyline. The
mythical tennis match mentioned above is a moneyline wager. Because two people
or teams that are competing against each other are rarely equally matched and
one person/team is generally perceived as being more likely to win, the prices
on the matches are set to where the team who is more likely to win (the
favorite) generally returns less than \$1 for every \$1 wagered, whereas the
team less likely to win (the underdog) returns more than \$1 for every \$1
wagered. In the example above, Federer is the favorite and Nadal is the
underdog.

### Pointspread {.unnumbered}

Another way to provide a method for equalizing the differences between the
favorites and the underdogs is to provide the underdog with a handicap. The
favorite has to win by a certain number of points in order to win (cover) the
pointspread wager, whereas the underdog does not need to win the match, but
rather just lose by less than the handicap (also referred to as cover) for
wagers on the underdog to be winners. Once the handicap is set, generally the
odds are the same whether a bettor chooses to wager on the favorite or the
underdog.

For example, in the 2026 Super Bowl (championship game of American football),
the Seattle Seahawks were the favorite and most bookmakers required a bettor to
risk \$2.30 to win \$1 if betting on the moneyline, whereas bettors who thought
the New England Patriots would win the game would receive about \$1.90 for
every \$1 risked on them to win. However, most bookmakers also established
Seattle as a 4.5 point favorite. This means that a bettor could bet on Seattle
to win by more than 4.5 points and only have to risk \$1.10 to win \$1.
Similarly a bettor could choose New England to win or if they lose, lose by
less than 4.5 points, but also have to risk \$1.10 to win \$1. This means that
for every \$11 the bookmaker attracted on Seattle -4.5, if he also attracted
\$11 on New England +4.5, he would guarantee a profit of \$1. If Seattle wins,
he would return the \$11 to the Seattle bettor plus their \$10 profit, while
keeping the \$11 from the New England bettor. The math is similar if New
England would win. The bookmaker would not want Seattle to win by exactly 4.5
points — since 4.5 is not a whole number, a push is impossible, which is one
reason half-point spreads are common.

The result of the game was that Seattle defeated New England 29-13. This
means that people who made moneyline wagers on Seattle were winners, and on
the pointspread Seattle bettors were also winners, as they won by 16 points,
well above the 4.5-point spread.

### Totals {.unnumbered}

The moneyline and pointspread examples above are what are called wagers on a
side, as a bettor chooses one team or the other. Another popular type of bet is
called betting on a total, or sometimes called an over/under bet. For a total
bet, a bookmaker sets a total number of points that would be scored in the
match, and then the bettor can wager on whether or not the actual total number
of points scored is over or under than total. Usually the amount risked vs
rewards is similar to the pointspread bet, where a bettor needs to risk \$1.10
to win \$1.00.

In the above example of the 2026 Super Bowl, most bookmakers had set the total
points scored at 45. In the actual game, only 42 points were scored (29+13).
So people that wagered on Under 45 won, while those who wagered on Over 45
lost their wagers.

On both totals and pointspread wagers, it is possible that the bookmaker could
get sided, or attract more bets on one outcome of a match than desired and have
an unbalanced market. To avoid this, a bookmaker will adjust the handicap (or
the total) if necessary to inspire the market to move in one direction or
another.

### Propositions (Prop Bets) {.unnumbered}

Prop bets are bets that are offered on something smaller than the general
outcome of the game. These might include will a hitter hit a home run or will a
pitcher achieve a certain number of strikeouts in baseball, or will a player
have over or under a certain number of rushing yards in football, or will a
goal be scored in the first ten minutes of a hockey game. These bets tend to be
offered at higher levels of juice to the player but allow for literally
hundreds of different bets to be placed on a single contest. These first became
popular in the Super Bowl to entice bettors to wager on many possible different
outcomes (and thus increase the amount of money wagered on the game), but with
the advent of mobile betting from your phone these are offered by some online
sportsbooks on virtually every game.

### Live Wagering {.unnumbered}

All the above examples are for wagers placed before the match starts, which is
how wagering has taken place for thousands of years. In the 21st century, the
advent of the Internet and computing technology has allowed for wagering to
occur during a particular competition. Market prices can be updated every few
seconds based upon the outcome of a particular play or point and how it might
affect the side or the total in that competition. An even more quick version of
live wagering can allow a bettor to wager on the outcome of a particular play.
This includes whether or not a player makes a free throw, a golfer makes a
putt, or a kicker makes a field goal.

The market for live wagering is virtually infinite. It can be difficult for
bookmakers to offer fair and accurate prices, and as such these are often more
heavily juiced than pre-game bets, but it allows for revenue to be earned by
the bookmaker throughout a contest. It also offers bettors opportunities to
increase the amount bet on an outcome or perhaps to hedge their bets based on
in-game injuries.

## Using Analytics to Set Market Prices

Many factors go into setting the market price in a contest. While of course
these would include some rating of each team's perceived ability in a vacuum,
there are additional factors that affect the predicted outcome of a particular
contest. In a team sport like football, it could be which team is the home
team, the surface of the field, the weather, the crew refereeing the game, any
current injured players, the time of year, the amount of rest since the team
previously played, staff changes, and results of previous contests between the
teams. In an individual sport like tennis, it could be the weather, the
surface, the conditions (indoor/outdoor), recent form, left handers vs right
handers, number of matches played recently, perceived crowd support, or even
travel and rest days.

As mentioned above, bookmakers can seek to guarantee profits by setting the
market prices such that equal financial obligations fall on both sides. However,
a bookmaker may also try to take advantage of market tendencies to maximize
profits. For example, studies have shown that in sports like football and
basketball that the betting public (otherwise known as squares) tends to prefer
favorites to underdogs and over to under in totals bets. This provides
opportunities for the bookmakers to adjust the prices in a way that might
attract more wagers on the favorites than their predictive models might
indicate, and similarly for the over in a totals wager than the under, allowing
for a potential for profit maximization.

Bookmakers employing the above strategy will be held in check by savvy
professional bettors (otherwise known as sharps) who are aware of these
tendencies of the public, spend many hours building their own predictive
models, and tend to place much larger wagers than your average square.
Bookmakers are aware of the sharps and will place limits on the amounts wagered
by an individual in order to limit exposure to sharp money. In more extreme
cases, some bookmakers will even refuse to take wagers from the sharpest of
sharps.

Much of the analytic modeling in predicting the outcomes of games and then
using them to set the market prices derives from other financial markets. For
example, Cantor Fitzgerald, a large financial services firm which was deeply
affected by the 9/11 attacks on the World Trade Center in New York, started a
division of the company called Cantor Gaming in the early 21st century to
provide market prices to Las Vegas casinos for live wagering markets, amongst
other products and services. While it was subsequently sold in 2020 to rival
bookmaker William Hill, these models and others are continuously refined to
take advantage of real and perceived market inefficiencies, much as a Wall
Street portfolio manager adjusts her trading strategies.

## Displaying and Calculating Market Prices

The prices on wagers are displayed in many different formats around the world,
but they can be converted to each other.

### American Odds {.unnumbered}

American odds are displayed in a format of either a positive or negative
number. A negative number would indicate how much money a bettor would have to
wager to win \$100. A positive number indicates an underdog and shows how much
money a bettor would win if he risked \$100. In the Super Bowl example above,
Seattle would have been listed as a -230 favorite, meaning that a bettor
would have to risk \$230 to win \$100 on the moneyline. New England would have
been listed as +190, meaning that a \$100 bet would have won the bettor \$190.

### Decimal Odds {.unnumbered}

Decimal odds are used in most of the world outside of the USA. They are given
in a format where a bettor can multiply the wager by the number listed to show
how much would be returned to the bettor if he wins his bet. This would
include the original wager plus the amount won. A bettor who wagers \$100 on a
team at 1.91 decimal odds and wins would get back \$191 if he wins (100*1.91)
and lose his wager of \$100 if he loses.

### Fractional Odds {.unnumbered}

Fractional odds are seen primarily in the UK and other commonwealth countries,
and also in horse and dog racing. They are displayed as a ratio of the amount
won to the amount risked, usually reduced as far as possible as a fraction. So
as the above examples in the pointspread and the totals wagers, where a bettor
must risk \$1.10 to win \$1.00, would be expressed as 10/11 in fractional terms.

### Probabilities {.unnumbered}

Probabilities are expressed as an implied probability of the outcome
happening. This is usually calculated as the amount risked divided by the
amount risked plus the amount won. So in the example above in fractional odds,
the probability would be 1.10/(1.10 + 1.00) = 52.38%. The implied probability
of course is not the actual probability, as there is juice built in for the
bookmaker. An additional calculation can be done to see the real probability
when the juice is removed.

### Odds Conversion Functions {.unnumbered}

Let's define some helper functions in Python to convert between these odds
formats. In R, one would use the `bettoR` package for these operations; in
Python we will write them ourselves.

In [ ]:
def american_to_decimal(american):
    """Convert American odds to decimal odds."""
    if american < 0:
        return -100 / american + 1
    else:
        return american / 100 + 1

def decimal_to_american(decimal):
    """Convert decimal odds to American odds."""
    if decimal < 2:
        return -100 / (decimal - 1)
    else:
        return (decimal - 1) * 100

def implied_prob(american):
    """Convert American odds to implied probability."""
    if american < 0:
        return abs(american) / (abs(american) + 100)
    else:
        return 100 / (american + 100)

def implied_odds(prob):
    """Convert a probability to American, decimal, and fractional odds."""
    decimal = 1 / prob
    american = decimal_to_american(decimal)
    fractional = decimal - 1
    return {"probability": prob, "american": american,
            "decimal": decimal, "fractional": round(fractional, 4)}

def hold_calc(odds1, odds2):
    """Calculate the bookmaker's hold (commission) given two American odds."""
    p1 = implied_prob(odds1)
    p2 = implied_prob(odds2)
    return p1 + p2 - 1

@tbl-equiv-odds shows the equivalent odds for the hypothetical tennis match
mentioned throughout this chapter.

In [ ]:
#| label: tbl-equiv-odds
#| tbl-cap: "Hypothetical tennis match odds in the various formats discussed in this chapter."

odds_df = pd.DataFrame({
    "Winner": ["Federer", "Nadal"],
    "American": ["-300", "+225"],
    "Decimal": [1.33, 3.25],
    "Fractional": ["1/3", "9/4"],
    "Probability": [0.75, 0.30]
})
odds_df

Of course we can also use our helper functions to convert one odds format to
another. The `implied_odds()` function starts with a probability and gives you
the odds in all other formats.

In [ ]:
implied_odds(0.75)

You may notice that the probabilities add up to more than 1, which usually
would violate every rule that we know about statistics. The reason for this is
that the probabilities also include the bookmaker's implied commission, which
we can calculate, for providing the market. In this case, this commission is
referred to as the hold.

In [ ]:
hold_calc(-110, -110)

This implies that if a sports book were to take two \$110 bets on this market,
one bet on team A and a similar bet on team B, both at -110 odds, the
bookmaker's commission is 4.545%. This is the total amount of the wagers (220)
less the amount paid back (100 winnings + 110 posted by the winning bettor) all
divided by the amount of the wagers taken (220), or 10/220 = 0.045.

## Straight Wagers vs. Parlays

All the wagers we have mentioned previously are called straight wagers. This
means that the wager is graded as a win or a loss based upon the outcome of a
single event, like a tennis match or a football game. However, another type of
wager with a higher payoff, albeit with a larger chance of losing, is called a
parlay. In a parlay, two or more outcomes are combined into a single wager. For
the bettor to win the parlay, she must correctly predict the outcome of both
events, otherwise it will be judged as a loss.

An example of a parlay bet that we now know would have been successful with
the benefit of hindsight would be for Seattle to win the 2026 Super Bowl AND
for the total points of the game to be under 45 points. We know that a bettor
could have wagered \$10 on the outcome of each event, and he would have won
both bets to win a total of \$13.40.

In [ ]:
side_wager = 10
side_multiplier = 1.43  # -230 in decimal odds
total_wager = 10
total_multiplier = 1.91  # -110 in decimal odds
payout = side_wager * side_multiplier + total_wager * total_multiplier
profit = payout - side_wager - total_wager
profit

However, if he had instead taken that \$20 and placed a parlay wager to win
both of these events, he would have profited much more, a total of \$34.80. The
amount of a parlay payout is the combination of both events happening, so we
need to multiply the probabilities together to find the outcome of two
uncorrelated events. The most straightforward way to calculate the fair parlay
odds is a four step process.

1. Make sure odds are converted to decimal odds.

In [ ]:
print(f"Seattle ML: {american_to_decimal(-230):.4f}")
print(f"Under 45:   {american_to_decimal(-110):.4f}")

2. Multiply the decimal odds together to get the parlay multiplier.

In [ ]:
parlay_mult = american_to_decimal(-230) * american_to_decimal(-110)
parlay_mult

3. Multiply the parlay multiplier by the amount wagered.

In [ ]:
parlay_mult * 20

4. Subtract the original amount to get the payout.

In [ ]:
parlay_mult * 20 - 20

So this \$20 parlay wager had American odds of +174 ((2.7406-1)*100) which
resulted in a profit of \$34.81. However, if either of these events did not
happen, the bettor would lose his entire \$20 wager, whereas if she wagered on
them as two separate \$10 events, he would have received a partial payout and
only lost his entire stake if he lost both wagers.

In general, parlays are much more profitable to the bookmaker due to these
combined odds, and much of a Nevada sportsbook's profits can be attributed to
parlays.

___

## Research Block

### Are sports wagering markets efficient? {.unnumbered}

Much of modern finance and its applications rely on the efficient market
hypothesis (EMH). Closely tied with Eugene Fama based on his 1970 review of
prediction literature (available at
[https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1540-6261.1970.tb00518.x](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1540-6261.1970.tb00518.x)),
EMH states that one can not make abnormal profits or "beat the market"
consistently over time. A market is thus efficient when the price of a
financial instrument reflects all available information, and that the
information is available to all market participants who are all able to
instantly absorb this information and use it in their investment strategies.
EMH holds that an individual might be able to beat the market return for a day,
week, month, or year, but eventually all participants will regress to the
market average returns.

As mentioned at the outset of this chapter, the authors view legalized and
regulated sports wagering markets as another financial market. Given that the
markets settle at the end of each event, and the bookmakers charge the vig as a
commission for offering wagers, if sports markets are truly efficient then
bettors should on average achieve a commission-free return on investment (ROI)
of zero, and then the market participants should all lose the vig over time.
For example, if the pointspread NFL wagering market is efficient, if a bettor
wagers on 100 NFL games, and wagers \$110 on each of them at the usual -110
pointspread market price, he should win 50 of those games, and as such win
\$5000 on the games he wins and lose \$5500 on the games he loses meaning that
he should lose \$500 for that \$11,000 wagered. There are short term variations,
but an efficient market would then create a -4.54% return (-500/11000). But
are the markets truly efficient?

In their 1997 *Journal of Finance* paper "Testing Market Efficiency: Evidence
From The NFL Sports Betting Market," available at
[https://www.jstor.org/stable/2329455](https://www.jstor.org/stable/2329455),
authors Philip Gray and Stephen Gray examine whether or not the betting market
is efficient. They gathered data on over 4200 NFL games played from 1976 to
1994, along with the closing line (the number of points that the favorite would
have to beat their opponent by in order to win the game when the game kicked
off) for each game. They examined several strategies which would be relatively
naive by today's analytics and yet discovered that there were several market
inefficiencies which also worked when tested against games not included in the
sample. For example, they found that by simply wagering on every home team who
was an underdog (*i.e.*, had points added to their total for the purpose of the
wager), a bettor could have earned a return of approximately +4.5%. However,
they also found that the success of many of these strategies dissipated over
time, presumably as more bettors learned about them and more bookmakers
adjusted their prices to give the home team underdog less of a handicap than
in past years.

So is the NFL pointspread betting market efficient? Many professional gamblers
will attest that it is the toughest market to consistently beat. As the bettors
build their own models which include extraneous factors like wind speed and
direction, the bookmakers also incorporate more and more information when
setting their market prices. An aspiring bettor would do well to focus on
different markets than the NFL pointspread.

___

## Kelly Criterion

When investing in traditional financial markets, participants often seek to
hold a diversified portfolio of assets. These might include stocks, bonds, real
estate, and today even cryptocurrency. Within each category, they might hold
several different types of stocks (such as tech companies, energy, and banks)
or real estate (commercial office space, warehouse, apartment complexes, and
single family houses for example). The idea behind holding these different
assets or different asset classes is to try to reduce risk of the portfolio's
value crashing because of an issue in one company (e.g. Enron) or one market
sector (such as Russian stocks in March 2022) through diversification. While
this limits the possibilities for extreme profits, it also limits the exposure
to extreme losses.

In sports wagering bettors also have to examine these criteria. A bettor's
bankroll (the total amount of money available to wager) could be doubled or
more by wagering it all on one game and winning. However, the bettor also has
the possibility to lose all of his money by wagering it on one game and
losing. The rational bettor would never wager all of her bankroll on any one
sporting event, but then is also faced with the problem of how much money to
risk on each wager while trying to keep the likelihood of several bad wagers
in a row bankrupting him to a minimum.

No stock picker is going to be correct 100% of the time, and no one is going
to be able to predict sporting events correctly 100% of the time. If someone
could predict them correctly 90% or 80% or even 70% of the time, no sports
book would take the person's wagers. Billy Walters, widely regarded as the best
and most successful sports wagerer in the world, was only correct about 57% of
the time (or 4 out of 7 times). If you are betting football and basketball
games on the pointspread, where generally the odds on either team are -110, you
need to be correct 52.38% of the time (11 times out of 21) to break even.
Given these odds, professional sports wagerers seek to be correct in the
54-56% range. The nature of which random events occurs also ensures that in a
long run, wagerers with a 55% success rate will also have periods of time
where they perform well above 55% and well below 55%. While performing above
55% may be great, performing below it for longer periods of time creates a risk
of ruin, something the mathematician J.L. Kelly modeled and provides advice on
how to minimize risk of ruin while maximizing returns.

The formula for Kelly Criterion is

$$
F = p-\frac{q}{b} = p-\frac{1-p}b
$$

where $F$ is the fraction of your bankroll to wager, $p$ is the probability of
a win, $q$ is the probability of a loss (1-probability of a win) and $b$ is
the proportion of the bet gained by winning (also equal to the decimal odds on
a bet -- 1).

So if you have a \$100 bankroll, and someone is offering you even money bets
(*e.g.* American Odds +100 or decimal odds of 2.00) on flipping a coin and you
know the coin is weighted such that it lands on heads 60% of the time, how
much should you wager?

In [ ]:
0.6 - (0.4 / 1)

The chances that you would go bankrupt on your first five wagers in this case
are $0.4^5$, or approximately 1% of the time. Your average outcome would be
that you would win three of these flips and lose two of them, so your bankroll
would increase on average by 20%.

It is unlikely that anyone would find this example in the real world, so let's
apply this math to the pointspread sports wagering examples mentioned above.
Let's say that we believe we will win a particular pointspread wager 55% of
the time, and the odds being offered are -110 American (1.91 decimal). What
percentage of your bankroll should you wager on that contest?

In [ ]:
0.55 - (0.45 / 0.91)

Let's define a `kelly()` function which allows us to calculate these
percentages more easily.

In [ ]:
def kelly(win_prob, american_odds):
    """Calculate the Kelly Criterion fraction of bankroll to wager."""
    decimal = american_to_decimal(american_odds)
    b = decimal - 1
    return win_prob - (1 - win_prob) / b

In [ ]:
print(f"Kelly fraction (55% win, -110 odds): {kelly(0.55, -110):.6f}")

The difficulty here is that we don't know with absolute certainty our
probability of picking correctly in the short term or the long term. We may
believe it is 55%, but if everyone could pick 55% of the time correctly then
the market makers would be out of business. What if it were a 54% chance of
success?

In [ ]:
print(f"Kelly fraction (54% win, -110 odds): {kelly(0.54, -110):.6f}")

What about 53% or 52%? The negative percentage associated with the 52% win
probability tells us not to bet!

In [ ]:
print(f"Kelly fraction (53% win, -110 odds): {kelly(0.53, -110):.6f}")
print(f"Kelly fraction (52% win, -110 odds): {kelly(0.52, -110):.6f}")

Because a bettor cannot wager a negative amount, he should not bet anything.
This makes sense since we know that a bettor must be successful 52.38% of the
time to break even in a -110 market.

There is a much longer proof of the Kelly Criterion but it is beyond the scope
of this text.

## Case Study: Stevin Smith, Tim Donaghy, and the issues of sports fixing

In financial markets, one concern is often if the market is fixed. By a market
being fixed, we mean that some market participants have access to information
that is not generally public and as such, the market price does not accurately
reflect all the information. In stocks and option markets, we generally refer
to this as inside information. Trading on inside information is illegal in the
US and most markets around the world and people can go to prison and/or pay
large fines for acting on that information.

As another financial market, the sports wagering industry is not immune to
these effects either. While there is nothing generally illegal about placing
wagers knowing that a player might be injured or something similar, it is
typically forbidden for players, coaches, and other employees to place wagers
within their own sport, and also to communicate such information to others on
the outside who may then participate in wagering markets. The most famous
recent case of gambling by players in these markets in the US is with the
baseball player Pete Rose, who was permanently banned from baseball in 1989
when it was found he had placed bets on baseball.

Another danger in these markets is when a player might intentionally not
perform at his or her best in order to affect the outcome of game. If a bettor
knows in advance that a player is intentionally not going to be performing
optimally, the bettor stands to make abnormal profits. In an individual sport
like tennis, it could be much easier for a player to intentionally lose a match
("throwing the match"), or to win by less than the game spread or lose by more
than the game spread ("Pointshaving"). In a team sport, it can be a bit more
difficult to have one player directly affect the outcome of a team game, as a
coach could decide whether or not to have a player play in the game or a
player may not be in enough of a position to affect the outcome of scoring.
Other cases you may have heard of include the 1919 "Black Sox" scandal in
major league baseball (a subplot of the 1989 movie Field of Dreams starring
Kevin Costner) and the Boston College basketball team in 1978-1979. Most
recently, University of Alabama baseball coach Brad Bohannon was fired in May
2023 after it was found he had ties to suspicious gamblers and fed information
about an injury to a starting pitcher for a game against LSU.

If a bettor could pay a player to throw a game, she can make abnormal profits.
If a bettor pays a tennis player \$5,000 to not win a match but can then
profit \$50,000 in the wagering markets by betting on that compromised player
to lose and be virtually assured that the person will indeed lose, then there
is great temptation to try to find a player who could be compromised. While
this is illegal in most locales for both the player and the bettor, the payoff
can be great if the player and bettor can get away with fixing that match.

Currently in the most popular professional sports in America, such as football,
baseball, basketball, and hockey, it is almost impossible to fix a game by
compromising a player. The reason is not morality or law enforcement, but
likely simple economics. The salaries have gotten so high that the bettor could
never pay the player enough money to throw a game and still be able to wager
enough money to make an abnormal profit.

| **Sport** | **Maximum Salary** | **Average Salary** | **Minimum Salary** |
|-----------|--------------------|--------------------|--------------------|
| MLB (2026) | \$61.88 MM | \$4.72MM | \$780K |
| NFL (2025) | \$60 MM | \$3.2MM | \$840K |
| NBA (2025-26) | \$59.61 MM | \$10.55MM | \$1.27MM |
| NHL (2025-26) | \$14 MM | \$3.5MM | \$775K |
: A survey of salaries in the MLB, NFL, NBA, and NHL. {#tbl-salaries}

Given the salaries of players today, it is hard to believe that a player would
jeopardize his criminal status and professional career for a \$5,000 or
\$10,000 payout from a bettor. If a bettor were to pay a player \$500,000 to
throw a game that might be tempting, but it is unlikely that the bettor would
be able to bet enough money on the game to make a profit, and certainly that
amount of money coming in would raise suspicions that something is likely
wrong with this market and market makers and law enforcement would then
investigate. But there are other potential weak points in the market.

Referees are paid far less than players, and they too have the ability to
affect the outcomes of games. While it may be obvious if a particular referee
is biased towards one team in getting that team to win, less obvious could be
totals bets. For example, in basketball, the more fouls that are called by a
referee, the more points will be scored on free throws and the less tight
defense will be played, which will then cause more points to be scored in the
normal course of the game. If a referee is calling many fouls, but calls them
equally on both teams, it may not appear that the game is fixed for one team
to win but a bettor who bets on the Over in the total bet with a referee who
is going to call an abnormal number of fouls would be able to profit.

Tim Donaghy was an NBA referee from 1994 to 2007. During those 13 seasons he
refereed approximately 60 games per year and also officiated a total of 20
playoff games during that time. However, Donaghy himself had a gambling problem
and was alleged to have bet tens of thousands of dollars on NBA games from
2005-2007 in an effort to get out of debt. Donaghy also admitted to having been
paid \$30,000 by organized crime affiliates to pass them information about
players' physical conditions and game situations. He was found guilty of
participating in the gambling scheme and sentenced to 15 months in prison and 3
years of probation following the prison term. While he was the one convicted,
he also claimed (but never proved) that several other referees had conspired to
fix games because it was in the interest of the NBA to fix them, such as to
extend playoff series or ensure that teams in larger media markets would win
games and thus make the NBA more money.

More recently, the NBA has faced integrity issues directly involving players.
In 2024, Jontay Porter, then a forward for the Toronto Raptors, was banned for
life by the NBA after it was found he had shared confidential medical
information with sports bettors who then wagered on his games ending early via
prop bets — and Porter himself had interests in the gambling proceeds. His case
represents one of the clearest modern examples of a player exploiting the
information asymmetry that comes with the expansion of player prop betting. In
a separate case, NBA guard Terry Rozier was suspended and remains under
investigation as of early 2026 for allegedly placing bets through associates
on basketball games, including games in which he participated; the matter is
still being litigated.

While the players in professional sports likely make too much money to be
corrupted by gamblers, collegiate sports are a different story. In the past,
intercollegiate athletes received only a free college education and meal and
travel expense reimbursement for their participation in representing their
schools on the athletic field. That landscape has shifted considerably: with
the introduction of Name, Image, and Likeness (NIL) rights in 2021 and the
landmark 2025 House v. NCAA settlement allowing schools to share revenue
directly with athletes, many college athletes in high-profile sports are now
being paid in excess of \$1 million dollars[^wager-2]. The annual men's college basketball 68-team single
elimination tournament in the spring, colloquially known as March Madness, is
the second biggest gambling event in the US behind only the Super Bowl. Over
\$16 billion was bet legally on the tournament in 2021 and much more was
wagered illegally. Many millions of dollars are bet on an average Saturday
during the college basketball season. In collegiate sports, football and
basketball get the most betting action, but football is much more difficult to
fix, given the number of players involved. With less betting action, it also
becomes more difficult to fix (a single large attempted bet was enough to
identify the abovementioned Alabama baseball issue). Combine the relative lack
of player remuneration with the amount of money wagered and you have the
opportunity for a player to be compromised.

[^wager-2]: NIL rights, introduced in 2021, allow college athletes to earn
money from endorsements and personal appearances. The 2025 House v. NCAA
settlement went further, permitting schools to directly share up to
approximately \$22 million per year in revenue with their athletes across all
sports, fundamentally changing the economic relationship between institutions
and the players who represent them.

Stevin Smith was such a player. From 1991 to 1994 he was a point guard for
Arizona State University's (ASU) basketball team. He was a two-time all-Pac 10
selection and as of 2023 still holds the school record for 3-point shots
attempted, 3-point shots made, and shares the school record for steals.
However, during his senior year he and fellow player Isaac Burton agreed to a
pointshaving scheme with a campus bookmaker, Benny Silman, to whom he
initially owed gambling debts from betting other sports. He pled guilty and
admitted to accepting \$20,000 in bribes while conspiring to shave points in a
game vs Oregon State and then more bribes for shaving points in 3 more games
in the 1993-1994 season and served one year in prison. The scheme cost him any
real chance to play in the NBA and he went undrafted. He played largely
overseas for 14 years, playing only 8 games in his career at the NBA level.

Smith's scheme was largely uncovered by the Las Vegas bookmakers, due to highly
unusual activity around the ASU-Washington game on March 5, 1994. As often
happens around college campuses, the rumor mill was running wild, and this time
it was about ASU men's basketball games being fixed. So many people around the
ASU campus and in criminal syndicates went to Las Vegas with the information
about the probability of a compromised player being in the game that the market
price changed 42 times, starting with ASU as a 10-point favorite over
Washington when the market opened but only a 1.5 point favorite when the game
tipped off. The game would have normally attracted about \$50,000 in action but
had over \$1 million bet on Washington. The Las Vegas bookmakers alerted the
FBI, and the ASU coach was notified at halftime that the FBI and the Pac-10
conference were looking into the game. Smith had an explosive performance in
the second half and ASU's lead went from 2 at halftime to 18 at game's end.
The people who were in on the scheme ultimately lost over one million dollars.

Given these possibilities for sports being fixed, consider the following
discussion questions.

1. What is the likelihood that professional sports are fixed today? Which
professional sports are most susceptible to being fixed?
2. How would you use analytics to ensure that a game is not fixed?
3. How do you compare the sports markets to financial markets in terms of their
fairness?
4. What have individual states done to try to prevent games from being fixed?
Is it effective? What else can be done?

## Daily Fantasy Sports

Another type of contest which became popular in the 1970s and 1980s was fantasy
sports, which allowed fans to act like their own general managers of a team. A
group of people would get together at the beginning of a season, choose players
for their team, and compete against each other throughout the year by measuring
the accumulation of various statistics by each player. The games evolved
throughout the year to allow for head-to-head competitions instead of simple
statistical accumulations. The most tedious part of these competitions was
keeping score, as it involved copying statistics from newspapers and tabulating
them without error. In the 1990s, Internet services evolved which hosted the
fantasy leagues and tabulated scores, either for a fee or free, but made the
process much easier.

In [ ]:
#| echo: false

display(Image(filename="fig/fantasy-trophy.jpeg", width=400))

Often these friendly competitions between friends would include an entry fee
which was redistributed to the winners. For example, ten people in a league
might each pay \$25 as an entry fee, and the winner of the league might get
\$125, the second place finisher \$75, and the third place finisher \$50. In
some circles, this was seen as tantamount to gambling, but in many others, it
was just seen as fun. When the United States Congress in 2006 passed the
Uniform Internet Gambling Enforcement Act (UIGEA), a law designed to combat
online poker and sports wagering sites, fantasy sports were specifically
exempted from UIGEA.

While the carveout from UIGEA was designed for season-long fantasy contests,
many of which were sponsored by media outlets and even the leagues themselves,
by the early 2010s many people desired quicker action. The idea of a fantasy
contest that was settled in one day rather than one season was born, and the
sites that sponsored these daily fantasy sports (DFS) contests became extremely
popular and competitive. To get an idea about the popularity and ferocity of
competition, on ESPN, the largest sports television network in the US, before
the 2015 NFL season an ad for either DraftKings or FanDuel, the two biggest
sites for DFS contests, ran approximately every 90 seconds.

With direct sports wagering now more available and more popular, DFS is not
quite as popular as before but still has quite the following in addition to
season long contests. As you might imagine since we are talking about it in
this book, analytics plays a huge role in choosing an optimal lineup to give a
player the best chance of winning the matchup. The sites know this and to some
extent support it. It is a balancing act for them to welcome new players and
make them feel comfortable while at the same time making it easier for big
volume players to participate as much as possible as the fees they pay with
each entry into a DFS contest generates revenue for the sites.

In [ ]:
#| echo: false

display(Image(filename="fig/dk-showdown.png", width=600))

One way the sites support analytics is to make the data for each day, league,
and contest downloadable. Each player on a team that is drafted has a
particular fantasy salary for that day, and each lineup submitted has a maximum
salary that is allowed, called the salary cap. This is to make the contests
more challenging than just selecting the six best goal scorers in a DFS hockey
contest and instead force the player to make choices when constructing his team
between player skill and maximum salary.

### DFS Lineup Construction {.unnumbered}

Ultimately, DFS lineup construction is an optimization problem. At a high
level, optimization is a mathematical technique of (usually):

1. maximizing or minimizing the value of a particular **objective** function,
based upon
2. **decision variables** that the person or computer solving the problem may
affect, subject to
3. **constraints** or limits that a decision maker must abide by.

To apply this to DFS, we might say that:

- the objective function is to maximize the number of fantasy points scored by
the team,
- the decision variables are which players to select,
- subject to the constraints of not exceeding the team salary cap. Each sport
requires additional roster considerations such as a maximum or minimum number
of players at each position or that play for a particular team in real life.

Our first example involves a Showdown contest on DraftKings. We use a Showdown
because it is one of the most basic contests, or optimization problems, on the
DraftKings platform. The contest involves choosing six players from the two
actual teams that are competing in a real-life game, subject to the following
constraints. First, we can only spend at most \$50,000 on the six players and
we must select at least one player from each of the two teams. Skaters and
goalies will accumulate points based on the distributions given in
@tbl-dk-skaters-pts and @tbl-dk-goalies-pts, respectively. In addition, we
will designate one player as the "Captain" and they will accumulate points at a
rate of 1.5 times the normal rate and their salary is 50% higher than their
non-Captain salary.

In [ ]:
#| label: tbl-dk-skaters-pts
#| tbl-cap: "DraftKings points (skaters) for an NHL showdown game."
#| echo: false

skater_pts = pd.DataFrame({
    "Statistics": ["Goal", "Assist", "Shot on Goal", "Blocked Shot",
                   "Shorthanded Goal", "Faceoff Won", "Shootout Goal",
                   "Hat Trick"],
    "Points": [8, 5, 2, 1.1, 2, 0.2, 1.5, 3]
})
skater_pts

In [ ]:
#| label: tbl-dk-goalies-pts
#| tbl-cap: "DraftKings points (goalies) for an NHL showdown game."
#| echo: false

goalie_pts = pd.DataFrame({
    "Statistics": ["Win", "Save", "Goal Allowed", "Shutout"],
    "Points": [4, 0.5, -1, 4]
})
goalie_pts

To make this example more concrete, we examine a hockey showdown and, in
particular, the data downloaded from DraftKings for Game 7 of the 2022 NHL
Playoffs between the New York Rangers and the Carolina Hurricanes. The data set
is given in the `isapy_data` module and is named `dk_nyr_car`. We will ignore
the `Captain` position in this initial solution. We will refer to this as the
simplified Showdown, and later we will see what happens when we introduce the
captain position and the full showdown problem.

In [ ]:
dk_nyr_car = isapy_data.load_data("dk_nyr_car")
df = dk_nyr_car[dk_nyr_car["roster_position"] == "FLEX"].copy().reset_index(drop=True)
df.head()

### Objective Function {.unnumbered}

We will define the objective to calculate the average points per game by
reading those past data values from the data frame, and later we will set that
to a maximum. The average points per game may not be a winning strategy in DFS
and we are only using it to illustrate the optimization problem. A professional
DFS player will likely spend a lot of time building models to predict each
player's expected points in order to use as their objective function.

### Constraints: Lineups, Salary, Teams {.unnumbered}

There are several constraints that we will implement:

1. We need to select exactly 6 players.
2. The total salary must not exceed \$50,000.
3. We need at least one player from each team.

### Solution {.unnumbered}

We will use the `PuLP` library to solve the optimization problem as a linear
program. `PuLP` allows us to define objectives, decision variables, and
constraints in a natural way. The decision variables are binary -- we either
include a player (1) or exclude them (0).

In [ ]:
players = df.index.tolist()
x = LpVariable.dicts("player", players, cat="Binary")

prob = LpProblem("DFS_Showdown_Simple", LpMaximize)

# Objective: maximize total avg_points_per_game
prob += lpSum([df.loc[i, "avg_points_per_game"] * x[i] for i in players])

# Constraint 1: exactly 6 players
prob += lpSum([x[i] for i in players]) == 6

# Constraint 2: salary cap of $50,000
prob += lpSum([df.loc[i, "salary"] * x[i] for i in players]) <= 50000

# Constraint 3: at least one player from each team
for team in df["team_abbrev"].unique():
    team_players = df[df["team_abbrev"] == team].index.tolist()
    prob += lpSum([x[i] for i in team_players]) >= 1

prob.solve(PULP_CBC_CMD(msg=0))
print(f"Status: {prob.status}, Optimal Points: {value(prob.objective):.2f}")

How do we identify the actual players that make up our team?

In [ ]:
selected_idx = [i for i in players if x[i].varValue == 1]
team = df.loc[selected_idx, ["name", "position", "salary", "team_abbrev"]]
team

While this is a valid solution for the model with the constraints we specified,
upon examination there are a few problems. First of all, we probably don't
want multiple goalies, particularly goalies from the same team, as it is
likely one of them will actually earn 0 points and will sit on the bench the
entire game. Let's add constraints to choose at most one goalie per team and
revise the solution.

In [ ]:
prob2 = LpProblem("DFS_Showdown_GoalieLimit", LpMaximize)
x2 = LpVariable.dicts("player", players, cat="Binary")

prob2 += lpSum([df.loc[i, "avg_points_per_game"] * x2[i] for i in players])
prob2 += lpSum([x2[i] for i in players]) == 6
prob2 += lpSum([df.loc[i, "salary"] * x2[i] for i in players]) <= 50000
for team_name in df["team_abbrev"].unique():
    team_players = df[df["team_abbrev"] == team_name].index.tolist()
    prob2 += lpSum([x2[i] for i in team_players]) >= 1
    # At most one goalie per team
    goalie_players = df[(df["team_abbrev"] == team_name) &
                        (df["position"] == "G")].index.tolist()
    prob2 += lpSum([x2[i] for i in goalie_players]) <= 1

prob2.solve(PULP_CBC_CMD(msg=0))
selected_idx2 = [i for i in players if x2[i].varValue == 1]
team2 = df.loc[selected_idx2, ["name", "position", "salary", "team_abbrev",
                                "avg_points_per_game"]]
team2

Does this solution spend all of our money?

In [ ]:
print(f"Total salary: ${team2['salary'].sum():,}")

Interestingly enough, it does not and this is not a problem at all. It meets
the constraint of being under the salary cap. There is still a problem with
this solution, though. Upon further inspection we see that the proposed
solution may include backup goalies who are unlikely to actually play. A more
realistic solution would be to filter out the non-starting goalies. The
starters for this game were Antti Raanta and Igor Shesterkin.

In [ ]:
df_new = df[(df["position"] != "G") |
            (df["name"].isin(["Antti Raanta", "Igor Shesterkin"]))].copy()
df_new = df_new.reset_index(drop=True)

players_new = df_new.index.tolist()
x3 = LpVariable.dicts("player3", players_new, cat="Binary")

prob3 = LpProblem("DFS_Showdown_Filtered", LpMaximize)
prob3 += lpSum([df_new.loc[i, "avg_points_per_game"] * x3[i]
                for i in players_new])
prob3 += lpSum([x3[i] for i in players_new]) == 6
prob3 += lpSum([df_new.loc[i, "salary"] * x3[i] for i in players_new]) <= 50000
for team_name in df_new["team_abbrev"].unique():
    team_players = df_new[df_new["team_abbrev"] == team_name].index.tolist()
    prob3 += lpSum([x3[i] for i in team_players]) >= 1

prob3.solve(PULP_CBC_CMD(msg=0))
selected_idx3 = [i for i in players_new if x3[i].varValue == 1]
team3 = df_new.loc[selected_idx3, ["name", "position", "salary",
                                    "team_abbrev", "avg_points_per_game"]]
team3

So while the math is correct, we should say that this solution is by no means
a guarantee to win a game. It is a very simple strategy and is highly
exploitable by professional DFS players. In fact, we would never advocate for
using two goalies in one of these games. We generally find a host of solutions
once we have added all constraints, *e.g.* use a single goalie, stack lines,
etc. So the questions remain, how can we find the next best? And the next? Or
the top 100? Update our constraints!

In [ ]:
results = []
prev_solutions = []

# Store the first solution
results.append({
    "players": team3["name"].tolist(),
    "points": team3["avg_points_per_game"].sum(),
    "salary": team3["salary"].sum()
})

for iteration in range(99):
    prob_iter = LpProblem(f"DFS_Iter_{iteration}", LpMaximize)
    x_iter = LpVariable.dicts(f"p_{iteration}", players_new, cat="Binary")

    prob_iter += lpSum([df_new.loc[i, "avg_points_per_game"] * x_iter[i]
                        for i in players_new])
    prob_iter += lpSum([x_iter[i] for i in players_new]) == 6
    prob_iter += lpSum([df_new.loc[i, "salary"] * x_iter[i]
                        for i in players_new]) <= 50000
    for team_name in df_new["team_abbrev"].unique():
        team_players = df_new[df_new["team_abbrev"] == team_name].index.tolist()
        prob_iter += lpSum([x_iter[i] for i in team_players]) >= 1

    # Exclude all previous solutions
    for prev_sol in prev_solutions:
        prob_iter += lpSum([x_iter[i] for i in prev_sol]) <= 5

    # Add the current best solution to exclusion list
    if iteration == 0:
        prev_solutions.append(selected_idx3)

    prob_iter.solve(PULP_CBC_CMD(msg=0))

    if prob_iter.status != 1:
        break

    sel = [i for i in players_new if x_iter[i].varValue == 1]
    prev_solutions.append(sel)
    sel_df = df_new.loc[sel]
    results.append({
        "players": sel_df["name"].tolist(),
        "points": sel_df["avg_points_per_game"].sum(),
        "salary": sel_df["salary"].sum()
    })

print(f"Generated {len(results)} unique lineups")
print(f"Best points: {results[0]['points']:.2f}")
print(f"Worst points: {results[-1]['points']:.2f}")

### Actual Showdown {.unnumbered}

Now that we know how to implement and solve an optimization problem in Python,
consider the full Showdown game. For this problem, we need to add in a
constraint to select a team captain and adjust the captain's salary and
expected points accordingly. The mechanics are essentially the same once we
add in this information.

In [ ]:
df_full = dk_nyr_car[
    (dk_nyr_car["position"] != "G") |
    (dk_nyr_car["name"].isin(["Antti Raanta", "Igor Shesterkin"]))
].copy().reset_index(drop=True)

# Create adjusted points for captains (1.5x)
df_full["new_pts"] = np.where(
    df_full["roster_position"] == "CPT",
    1.5 * df_full["avg_points_per_game"],
    df_full["avg_points_per_game"]
)

players_full = df_full.index.tolist()
x_full = LpVariable.dicts("full", players_full, cat="Binary")

prob_full = LpProblem("DFS_Full_Showdown", LpMaximize)
prob_full += lpSum([df_full.loc[i, "new_pts"] * x_full[i]
                    for i in players_full])

# Exactly 6 players
prob_full += lpSum([x_full[i] for i in players_full]) == 6

# Salary cap
prob_full += lpSum([df_full.loc[i, "salary"] * x_full[i]
                    for i in players_full]) <= 50000

# At least one player per team
for team_name in df_full["team_abbrev"].unique():
    team_players = df_full[df_full["team_abbrev"] == team_name].index.tolist()
    prob_full += lpSum([x_full[i] for i in team_players]) >= 1

# Exactly one captain
cpt_players = df_full[df_full["roster_position"] == "CPT"].index.tolist()
prob_full += lpSum([x_full[i] for i in cpt_players]) == 1

# Each player can appear at most once (as CPT or FLEX, not both)
for player_name in df_full["name"].unique():
    player_rows = df_full[df_full["name"] == player_name].index.tolist()
    prob_full += lpSum([x_full[i] for i in player_rows]) <= 1

prob_full.solve(PULP_CBC_CMD(msg=0))
selected_full = [i for i in players_full if x_full[i].varValue == 1]
team_full = df_full.loc[selected_full, ["name", "position", "salary",
                                         "team_abbrev", "roster_position",
                                         "avg_points_per_game"]]
team_full

## Lab: Scraping Odds Data to Build Models

In the Research Block for this section, we mentioned the research that Gray and
Gray did which uncovered some inefficiencies in the NFL betting markets. They
did this by obtaining data for the closing lines for NFL games and then
analyzing them against the final scores of the games played. They could have
(and perhaps did!) used this data set not just to publish an academic paper,
but perhaps try to profit in sports betting markets. What if you wanted to
analyze past betting data yourself?

Finding betting data from games (or matches) in the past can be difficult.
Some places who store it charge high monthly fees to access the data, as they
assume that the users are going to be bettors building models. But there is one
place that the authors know of that provide the data, Sportsbook Reviews
Online. While primarily an affiliate marketer (which the authors neither
endorse nor condemn), they also maintain an archive of odds at
[https://www.sportsbookreviewsonline.com/scoresoddsarchives/](https://www.sportsbookreviewsonline.com/scoresoddsarchives/).
We can scrape this data and do our own analyses.

1. Go to the NFL data link on the above page and scrape the data from the
2025-2026 NFL season and write it to a data frame `df`. Recall the
webscraping material from a previous chapter.

In [ ]:
#| eval: false

import requests
from bs4 import BeautifulSoup
from io import StringIO

url = "https://www.sportsbookreviewsonline.com/scoresoddsarchives/nfl-odds-2025-26/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table")
df = pd.read_html(StringIO(str(table)))[0]
df.columns = df.iloc[0]
df = df.iloc[1:]

2. What do you notice when exploring the data? There are the obvious column
names, but some of them may be unfamiliar to you.

- "Rot" is short for "Rotation Number", essentially a key for the team/game
when combined with the date.
- VH is Visitor or Home.
- 1st/2nd/3rd/4th are the scoring by that team in those quarters, and Final is
the final score.
- Open and Close are the opening and closing lines. The total is listed next to
the underdog, and the spread is listed next to the favorite.
- ML is the Money Line at close.
- 2H is the second half line.

3. If you want to use the data for analysis, what tasks need to be performed to
clean the data?

Sometimes the team that is the favorite when the betting opens is an underdog
by the time the betting closes. Also, if the teams are evenly matched, the
point spread will be written as "pk", short for "pick 'em", or evenly rated.
It is equivalent to a spread of 0, so those need to be converted.

Write code that will check for these anomalies and correct for them before
writing them into a data frame for analysis.

4. Examine the data for trends. What do you notice, if anything? Perhaps
compare it to data from other years to identify trends.

## Conclusion

Many of the analytic methods we have discussed throughout this text can be
applied to sports wagering. Through the use of mathematics, a bettor can give
himself the best possible chances for success.

The issue with developing complex analytic models for success in sports
wagering is that even if one can predict the outcomes of games often enough to
be successful, he will find it difficult to find a bookmaker willing to accept
his wagers in a high enough amount to gain long term profits and income that is
worth the time invested in building the models. There are of course different
tricks and techniques that professional bettors use to get their wagers down
and accepted, but these are also beyond the scope of this chapter.

## Exercises

1. Write a function that converts American odds to decimal odds.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

def us_to_decimal_odds(us_odds):
    if us_odds <= -100:
        return -100 / us_odds + 1
    else:
        return us_odds / 100 + 1

# Test
print(us_to_decimal_odds(-110))  # 1.909...
print(us_to_decimal_odds(150))   # 2.5

2. In the 2023 Super Bowl, the underdog Kansas City Chiefs defeated the
Philadelphia Eagles by a score of 38-35. Before the game, the bookmakers had
installed Philadelphia as a 1.5 point favorite, and the total settled at 51.5.
What does this imply the predicted final score would be? What could a bettor do
with this information?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

sb_total_points = 51.5
phi_points = sb_total_points / 2 + 1.5 / 2
kc_points = sb_total_points - phi_points
print(f"The implied final score of the 2023 Super Bowl, by the bookmakers, "
      f"is {phi_points} to {kc_points}.")

3. In many parts of the country, parlay cards are used in the legal (and
illegal) betting markets. A parlay card commonly will show point spreads and
totals for all the NFL games for a given weekend or NBA games for a given day,
and a bettor will select any number of them (usually somewhere between 3 and
10) and make a wager which will then return a certain amount. In the attached
graphic, a 4 team parlay would pay 10 times the wager, and a 5 team parlay
would pay 15 times the wager. These are usually huge moneymakers for the
person offering the cards because the hold is so high.

In [ ]:
#| echo: false
#| eval: false

display(Image(filename="fig/lucky-lenny.jpeg", width=400))

a. What is the fair return for every dollar wagered on a 4 team parlay if the
teams are evenly matched and both offered at even money (i.e., +100 odds,
without juice)?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

print(f"Fair return: {1 / (0.5 ** 4):.0f} to 1")

b. What is the fair return for every dollar wagered on a four team parlay if
the teams are evenly matched but are both offered at -110 odds by the
bookmaker (standard juice)?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

print(f"Fair return: {1 / ((110 / 210) ** 4):.5f} to 1")

c. Using your answer to (b) above, what is the implied probability of winning
all the games on a four team parlay?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

fair_return = 1 / ((110 / 210) ** 4)
print(f"Implied probability: {1 / fair_return:.6f}")

d. If a person offers parlay cards with exactly 4 games on them to 16 people
and each one of them wagers \$1 and all 16 combinations are taken (2 outcomes
for 4 games = $2^4$ combinations, or 16), what should the parlay card offerer
expect to profit? Note that a winning parlay card returns 10 for 1 and not 10
to 1, so the original wager in a winner is not returned in 10 for 1 (and it
functions like 9 to 1).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

profit = 16 - 10
print(f"Expected profit: ${profit}")

4. In NFL and NBA games, point spread wagers are usually made on either team at
-110 odds. On a given NFL Sunday, where a bookmaker might attract \$50 million
in bets, equally divided among all teams on games at -110 odds, how much
profit would be lost by going to -105 odds? How much more business would that
bookmaker have to attract in order to make the same profit as they would have
made at -110 odds?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

profit_df = pd.DataFrame({"odds": [-110, -105]})
profit_df["profit"] = 50_000_000 / 2 * (1 + 100 / profit_df["odds"])
profit_df

5. Read in the DraftKings data for the 2022 NHL playoff game between the
Edmonton Oilers and the Colorado Avalanche (provided in the `isapy_data`
module, `dk_edm_col`). Filter to just include the non-Captain players in order
to keep things simple. Compute a scatter plot showing `salary` on the x axis
and `avg_points_per_game` on the y axis. Color the points by `team_abbrev`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

dk_edm_col = isapy_data.load_data("dk_edm_col")
df_ex = dk_edm_col[dk_edm_col["roster_position"] != "CPT"].copy()

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("Set1", 2)
for i, team_name in enumerate(sorted(df_ex["team_abbrev"].unique())):
    subset = df_ex[df_ex["team_abbrev"] == team_name]
    ax.scatter(subset["salary"], subset["avg_points_per_game"],
               color=palette[i], label=team_name, alpha=0.7)
ax.set_xlabel("Salary")
ax.set_ylabel("Average Points per Game")
ax.set_title("DK Fantasy Salary vs Fantasy PPG")
ax.legend(title="Team")
plt.show()

6. Find the DraftKings 6-player Showdown team having the highest (aggregate)
`avg_points_per_game` subject to: (1) at least one player from each team, (2)
the team salary must be less than \$50,000, and (3) six players total. Filter
out the non-starting goalies (the starters were Darcy Kuemper and Mike Smith).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

dk_edm_col = isapy_data.load_data("dk_edm_col")
df_ex = dk_edm_col[dk_edm_col["roster_position"] == "FLEX"].copy()
df_ex = df_ex[(df_ex["position"] != "G") |
              (df_ex["name"].isin(["Darcy Kuemper", "Mike Smith"]))].reset_index(drop=True)

players_ex = df_ex.index.tolist()
x_ex = LpVariable.dicts("ex6", players_ex, cat="Binary")

prob_ex = LpProblem("Exercise_6", LpMaximize)
prob_ex += lpSum([df_ex.loc[i, "avg_points_per_game"] * x_ex[i]
                  for i in players_ex])
prob_ex += lpSum([x_ex[i] for i in players_ex]) == 6
prob_ex += lpSum([df_ex.loc[i, "salary"] * x_ex[i] for i in players_ex]) <= 50000
for team_name in df_ex["team_abbrev"].unique():
    team_players = df_ex[df_ex["team_abbrev"] == team_name].index.tolist()
    prob_ex += lpSum([x_ex[i] for i in team_players]) >= 1

prob_ex.solve(PULP_CBC_CMD(msg=0))
selected_ex = [i for i in players_ex if x_ex[i].varValue == 1]
df_ex.loc[selected_ex]

7. Suppose that you are particularly bullish on the Avalanche given that this
game is in Denver. Find the best team (as above) that has at least four
Avalanche players and report the expected points.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

x_ex7 = LpVariable.dicts("ex7", players_ex, cat="Binary")
prob_ex7 = LpProblem("Exercise_7", LpMaximize)
prob_ex7 += lpSum([df_ex.loc[i, "avg_points_per_game"] * x_ex7[i]
                   for i in players_ex])
prob_ex7 += lpSum([x_ex7[i] for i in players_ex]) == 6
prob_ex7 += lpSum([df_ex.loc[i, "salary"] * x_ex7[i]
                   for i in players_ex]) <= 50000

# At least 4 Avalanche players
col_players = df_ex[df_ex["team_abbrev"] == "COL"].index.tolist()
prob_ex7 += lpSum([x_ex7[i] for i in col_players]) >= 4

edm_players = df_ex[df_ex["team_abbrev"] == "EDM"].index.tolist()
prob_ex7 += lpSum([x_ex7[i] for i in edm_players]) >= 1

prob_ex7.solve(PULP_CBC_CMD(msg=0))
selected_ex7 = [i for i in players_ex if x_ex7[i].varValue == 1]
df_ex.loc[selected_ex7]

8. The lineup in problem 6 does not include Cale Makar, but assume you really
want him on your team. Make sure that both Cale Makar and Nathan MacKinnon are
included. You may remove the constraint that the team must include four
Avalanche players.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

x_ex8 = LpVariable.dicts("ex8", players_ex, cat="Binary")
prob_ex8 = LpProblem("Exercise_8", LpMaximize)
prob_ex8 += lpSum([df_ex.loc[i, "avg_points_per_game"] * x_ex8[i]
                   for i in players_ex])
prob_ex8 += lpSum([x_ex8[i] for i in players_ex]) == 6
prob_ex8 += lpSum([df_ex.loc[i, "salary"] * x_ex8[i]
                   for i in players_ex]) <= 50000
for team_name in df_ex["team_abbrev"].unique():
    team_players = df_ex[df_ex["team_abbrev"] == team_name].index.tolist()
    prob_ex8 += lpSum([x_ex8[i] for i in team_players]) >= 1

# Must include Makar and MacKinnon
must_include = df_ex[df_ex["name"].isin(
    ["Cale Makar", "Nathan MacKinnon"])].index.tolist()
prob_ex8 += lpSum([x_ex8[i] for i in must_include]) == 2

prob_ex8.solve(PULP_CBC_CMD(msg=0))
selected_ex8 = [i for i in players_ex if x_ex8[i].varValue == 1]
df_ex.loc[selected_ex8]

9. Using this same data set, find the top legitimate DraftKings Showdown
lineup as we did earlier in the chapter. Again, only consider the starting
goalies.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

dk_edm_col = isapy_data.load_data("dk_edm_col")
df_ex9 = dk_edm_col[
    (dk_edm_col["position"] != "G") |
    (dk_edm_col["name"].isin(["Darcy Kuemper", "Mike Smith"]))
].copy().reset_index(drop=True)

df_ex9["new_pts"] = np.where(
    df_ex9["roster_position"] == "CPT",
    1.5 * df_ex9["avg_points_per_game"],
    df_ex9["avg_points_per_game"]
)

players_ex9 = df_ex9.index.tolist()
x_ex9 = LpVariable.dicts("ex9", players_ex9, cat="Binary")

prob_ex9 = LpProblem("Exercise_9", LpMaximize)
prob_ex9 += lpSum([df_ex9.loc[i, "new_pts"] * x_ex9[i] for i in players_ex9])
prob_ex9 += lpSum([x_ex9[i] for i in players_ex9]) == 6
prob_ex9 += lpSum([df_ex9.loc[i, "salary"] * x_ex9[i]
                   for i in players_ex9]) <= 50000

for team_name in df_ex9["team_abbrev"].unique():
    team_players = df_ex9[df_ex9["team_abbrev"] == team_name].index.tolist()
    prob_ex9 += lpSum([x_ex9[i] for i in team_players]) >= 1

# Exactly one captain
cpt = df_ex9[df_ex9["roster_position"] == "CPT"].index.tolist()
prob_ex9 += lpSum([x_ex9[i] for i in cpt]) == 1

# Each player at most once
for name in df_ex9["name"].unique():
    name_rows = df_ex9[df_ex9["name"] == name].index.tolist()
    prob_ex9 += lpSum([x_ex9[i] for i in name_rows]) <= 1

prob_ex9.solve(PULP_CBC_CMD(msg=0))
selected_ex9 = [i for i in players_ex9 if x_ex9[i].varValue == 1]
df_ex9.loc[selected_ex9, ["name", "position", "salary", "team_abbrev",
                           "roster_position", "avg_points_per_game"]]